# NeuroWrite PT — Experimento 1: EMNIST Letters
**Projeto-terça (independente do EPA) · Python / Google Colab / GitHub Actions**

**Problema:** classificação supervisionada de imagens de caracteres manuscritos A–Z (26 classes).  
**Objetivo:** comparar SVM e MLP por validação cruzada e conjunto de teste independente, com métricas e gráficos.  
**Fonte primária:** [NIST — EMNIST](https://www.nist.gov/itl/products-and-services/emnist-dataset).  
**Cópia de distribuição dos arquivos IDX gzip:** [Royc30ne/emnist-letters](https://huggingface.co/datasets/Royc30ne/emnist-letters).

> **Escopo:** fase exploratória **pública**, sem dados do grupo. A variante Letters combina exemplos maiúsculos e minúsculos em 26 classes. **Não contém Ç ou vogais acentuadas** e não comprova reconhecimento de frases. O experimento manuscrito do grupo com 38 classes será feito separadamente.

Este notebook também pode ser executado por uma GitHub Action; o próprio arquivo com saídas será disponibilizado em artefatos da execução. Antes de entregar ao professor, manter os resultados persistidos no `.ipynb`.

## Resumo / introdução
O reconhecimento de caracteres manuscritos permite estudar como modelos aprendem padrões visuais de diferentes pessoas. O conjunto EMNIST fornece imagens de 28 × 28 pixels de caracteres. Aqui comparamos **SVM de kernel RBF** e **rede neural MLP** sem usar texto ou imagens sintéticas inventadas. O resumo quantitativo será escrito após os resultados.

## Objetivo e questões
1. Explorar a distribuição das classes e exemplos do conjunto público.
2. Treinar SVM e MLP com diferentes hiperparâmetros.
3. Comparar **acurácia, precisão macro, recall macro e F1 macro**.
4. Identificar confusões entre letras e limitações da amostra.
5. Preparar uma base técnica para a coleta local (que possui rótulos/estilo diferentes).

**Hipótese testável:** os classificadores apresentarão desempenhos diferentes em letras visualmente semelhantes. Nenhum resultado é assumido antecipadamente.

## Desenvolvimento — Fonte, características e download
O EMNIST *Letters* possui 26 classes de letras (maiúsculas e minúsculas fundidas), com imagens 28 × 28 em escala de cinza. Para tornar o treinamento viável no Colab/Actions, selecionamos uma quantidade **determinística e equilibrada por classe** tanto do conjunto oficial de treino quanto do conjunto de teste. Não misturamos as duas divisões.

**Entrada (features):** 784 intensidades numéricas (pixels de 0 a 1).  
**Alvo:** letra A–Z (26 categorias).  
**Repetibilidade:** semente aleatória 42.  
**Limite:** tamanho reduzido da amostra; não interpretar a métrica como benchmark de todo o EMNIST.

Os arquivos IDX têm uma orientação histórica particular; os pixels são transpostos para exibição legível. A mesma transformação é aplicada consistentemente aos exemplos de treino e teste.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from time import sleep
import gzip
import json
import struct
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import joblib

SEMENTE = 42
LETRAS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
TREINO_POR_CLASSE = 100
TESTE_POR_CLASSE = 20
BASE = "https://huggingface.co/datasets/Royc30ne/emnist-letters/resolve/main/"
CACHE = Path("data/public/emnist_letters")
SAIDA = Path("results/emnist")
CACHE.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)
np.random.seed(SEMENTE)

def baixar(nome):
    destino = CACHE / nome
    if destino.exists() and destino.stat().st_size > 128:
        return destino
    url = BASE + nome + "?download=true"
    for tentativa in range(3):
        try:
            print("Baixando", nome)
            requisicao = Request(url, headers={"User-Agent": "NeuroWrite-PT/1.0"})
            with urlopen(requisicao, timeout=120) as resposta:
                with destino.open("wb") as f:
                    while True:
                        bloco = resposta.read(1024 * 1024)
                        if not bloco:
                            break
                        f.write(bloco)
            return destino
        except Exception:
            destino.unlink(missing_ok=True)
            if tentativa == 2:
                raise
            sleep(2 ** tentativa)

def ler_idx(nome):
    caminho = baixar(nome)
    with gzip.open(caminho, "rb") as arq:
        magic = struct.unpack(">I", arq.read(4))[0]
        if magic not in (2049, 2051):
            raise ValueError(f"Formato IDX inesperado em {nome}: {magic}")
        dimensoes = magic & 0xFF
        formato = ">" + "I" * dimensoes
        shape = struct.unpack(formato, arq.read(4 * dimensoes))
        dados = np.frombuffer(arq.read(), dtype=np.uint8).reshape(shape)
    return dados.copy()

def carregar_subset(particao, por_classe):
    imagens = ler_idx(f"emnist-letters-{particao}-images-idx3-ubyte.gz")
    rotulos = ler_idx(f"emnist-letters-{particao}-labels-idx1-ubyte.gz")
    assert imagens.shape[0] == rotulos.shape[0]
    assert imagens.shape[1:] == (28, 28)
    assert rotulos.min() >= 1 and rotulos.max() <= 26
    rng = np.random.default_rng(SEMENTE + (0 if particao == "train" else 1))
    indices = np.concatenate([
        rng.choice(np.flatnonzero(rotulos == classe), size=por_classe, replace=False)
        for classe in range(1, 27)
    ])
    rng.shuffle(indices)
    # Transposição documentada das imagens IDX do EMNIST (orientação visual).
    imgs = imagens[indices].transpose(0, 2, 1)
    X = imgs.reshape(-1, 784).astype(np.float32) / 255.0
    y = rotulos[indices] - 1
    return X, y, imgs

X_treino, y_treino, imagens_treino = carregar_subset("train", TREINO_POR_CLASSE)
X_teste, y_teste, imagens_teste = carregar_subset("test", TESTE_POR_CLASSE)
assert len(X_treino) == 26 * TREINO_POR_CLASSE
assert len(X_teste) == 26 * TESTE_POR_CLASSE
print("Treino:", X_treino.shape, "Teste:", X_teste.shape)
print("Classes:", len(np.unique(y_treino)), "Intervalo dos pixels:", (X_treino.min(), X_treino.max()))

### Análise exploratória
As figuras seguintes são geradas a partir das **observações reais baixadas**. O recorte equilibrado é deliberado: deixa a comparação entre as 26 classes mais simples, mas não representa o universo completo de caligrafias humanas.

In [ ]:
fig, axs = plt.subplots(2, 8, figsize=(13, 4))
rng = np.random.default_rng(SEMENTE)
escolhidos = rng.choice(len(imagens_treino), size=16, replace=False)
for ax, idx in zip(axs.ravel(), escolhidos):
    ax.imshow(imagens_treino[idx], cmap="gray")
    ax.set_title(LETRAS[y_treino[idx]])
    ax.axis("off")
plt.suptitle("Exemplos reais — EMNIST Letters (subconjunto de treino)")
plt.tight_layout()
plt.savefig(SAIDA / "amostras.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(12, 3))
pd.Series([LETRAS[i] for i in y_treino]).value_counts().reindex(LETRAS).plot.bar(ax=ax)
ax.set(title="Distribuição das classes — treino", ylabel="Quantidade", xlabel="Letra")
plt.tight_layout()
plt.savefig(SAIDA / "distribuicao.png", dpi=150)
plt.show()

### Seleção de features e validação
Todas as 784 intensidades dos pixels serão usadas nesta fase exploratória; não há seleção manual baseada no conjunto de teste. O escalonamento é ajustado **dentro** do pipeline e apenas nos folds de treinamento.

Dois algoritmos:
- **SVM RBF:** `C ∈ {1, 10}` (regularização).
- **MLP:** `hidden_layer_sizes ∈ {(64,), (128,)}` (arquitetura).

Usaremos **StratifiedKFold com três folds** no conjunto de treino para comparar parâmetros. O conjunto oficial de teste (amostrado) permanece intocado até a avaliação final. Em cada modelo, a escolha de hiperparâmetros é definida pela **média do F1 macro nos folds**, e não pela acurácia do teste.

**Não é o experimento por escritor:** o EMNIST usado aqui não dispõe de identidade individual confiável na cópia de distribuição. O estudo posterior com três autores utilizará `LeaveOneGroupOut`.

In [ ]:
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEMENTE)
configuracoes = {
    "SVM": (Pipeline([
        ("escala", StandardScaler()),
        ("modelo", SVC(kernel="rbf", gamma="scale"))
    ]), {"modelo__C": [1.0, 10.0]}),
    "MLP": (Pipeline([
        ("escala", StandardScaler()),
        ("modelo", MLPClassifier(
            random_state=SEMENTE, max_iter=150, early_stopping=True,
            n_iter_no_change=12
        ))
    ]), {"modelo__hidden_layer_sizes": [(64,), (128,)]})
}
linhas, predicoes_finais = [], {}
for nome, (pipeline, grade) in configuracoes.items():
    print("Treinando:", nome)
    busca = GridSearchCV(pipeline, grade, scoring="f1_macro", cv=kfold,
                         n_jobs=2, error_score="raise")
    busca.fit(X_treino, y_treino)
    pred = busca.predict(X_teste)
    predicoes_finais[nome] = pred
    linhas.append({
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, pred),
        "precisao_macro": precision_score(y_teste, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_teste, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_teste, pred, average="macro", zero_division=0),
        "cv_f1_macro": busca.best_score_,
        "hiperparametros": str(busca.best_params_)
    })
    joblib.dump(busca.best_estimator_, SAIDA / f"modelo_{nome.lower()}.joblib", compress=3)
    print("CV F1 macro:", round(busca.best_score_, 4))
resultado = pd.DataFrame(linhas).sort_values("f1_macro", ascending=False)
display(resultado)
resultado.to_csv(SAIDA / "metricas.csv", index=False)
(SAIDA / "metadados.json").write_text(json.dumps({
    "dataset": "EMNIST Letters",
    "fonte": BASE,
    "rotulos": "1..26 (A-Z), variante Letters com caixa alta/baixa mescladas",
    "treino_por_classe": TREINO_POR_CLASSE,
    "teste_por_classe": TESTE_POR_CLASSE,
    "seed": SEMENTE,
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "metricas": resultado.to_dict(orient="records")
}, indent=2, ensure_ascii=False), encoding="utf-8")

### Comparação dos resultados
Os resultados abaixo diferenciam **acurácia global** e **F1 macro**, para não disfarçar erros em categorias individuais. As matrizes de confusão exibem quais letras são mais confundidas. A interpretação da diferença entre as métricas deve considerar a quantidade reduzida de exemplos selecionados.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
resultado.set_index("modelo")[["acuracia", "f1_macro", "cv_f1_macro"]].plot.bar(ax=ax)
ax.set(ylabel="Pontuação (0 a 1)", title="Comparação de modelos — EMNIST", ylim=(0, 1.05))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(SAIDA / "comparacao.png", dpi=150)
plt.show()

for nome, pred in predicoes_finais.items():
    fig, ax = plt.subplots(figsize=(10, 9))
    ConfusionMatrixDisplay.from_predictions(
        y_teste, pred, labels=list(range(26)),
        display_labels=LETRAS, cmap="Blues", xticks_rotation=90,
        values_format="d", colorbar=False, ax=ax
    )
    ax.set_title(f"Matriz de confusão — {nome}")
    plt.tight_layout()
    plt.savefig(SAIDA / f"matriz_confusao_{nome.lower()}.png", dpi=140)
    plt.show()
    print(f"Relatório de classificação — {nome}")
    print(classification_report(y_teste, pred, labels=list(range(26)),
                                target_names=LETRAS, zero_division=0))

## Conclusão e limitações

Ao apresentar o trabalho, interpretar **os valores efetivamente calculados acima**, sem presumir vencedor antecipadamente.

Limitações relevantes:
1. Este é um experimento público inicial, limitado a **A–Z** e a um recorte de amostras.
2. A variante EMNIST *Letters* une maiúsculas e minúsculas na mesma classe, diferindo da coleta do grupo (caixa alta apenas).
3. Não há dados dos três integrantes neste treinamento; **não houve validação por escritor**.
4. O dataset original não inclui acentos do português, cedilha nem frases completas.
5. Classificar letras isoladas não resolve automaticamente segmentar palavras manuscritas.

**Próxima fase:** coletar folhas piloto, extrair imagens e executar o notebook principal em `notebooks/NeuroWrite_PT.ipynb`, avaliando os três escritores separadamente.

## Referências
- COHEN, G. et al. EMNIST: An Extension of MNIST to Handwritten Letters. 2017. https://arxiv.org/abs/1702.05373
- NIST. EMNIST Dataset. https://www.nist.gov/itl/products-and-services/emnist-dataset
- SCIKIT-LEARN. SVC, MLPClassifier, GridSearchCV e métricas de classificação. https://scikit-learn.org/stable/
- Mirror de distribuição dos arquivos IDX usados: https://huggingface.co/datasets/Royc30ne/emnist-letters